# The Heatwave Hustle: How Chicagoans Move in Extreme Weather

## Narrative Hook

When the temperature climbs, so do tempers. Chicago’s summer heat waves are infamous—but do they change how people move around the city?

## Tension & Stakes

Weather impacts everything. But **how much does heat change Uber demand, tipping habits, and travel patterns**?

* Do people take **more** Ubers when it’s dangerously hot?
* Do they tip more because they _empathize_ with a driver suffering in the heat?
* Or do they tip _less_ because they’re already miserable and irritable?

## Reveal & Visualization Opportunities

* **Uber Trips vs. Temperature**: Does ridership spike on 90°F+ days?
* **Tipping Behavior in Extreme Weather**: Are people kinder in heat waves or stingier?
* **Surge Pricing vs. Weather**: Does Uber jack up fares when people are desperate?

## Resolution

A hot summer in Chicago isn’t just about sweat—it’s about _how_ the city adapts. Does extreme heat create generosity, stress, or indifference?

## Why This Works

It taps into **weather**: something everyone experiences and has opinions on.
It links to **human behavior**: heat stress, money decisions, social dynamics.
It tests _assumptions_: do people become better or worse versions of themselves in extreme conditions?


In [2]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

rides = pd.read_csv('data/rides.csv', dtype={'pickup_zip': 'str', 'dropoff_zip': 'str'}, parse_dates=['ride_start', 'ride_end'])
weather = pd.read_csv('data/chicago_hourly_weather.csv', parse_dates=['datetime'])
rides = rides.sort_values('ride_start')
rides['tipped?'] = rides['tip'] != 0
weather = weather.sort_values('datetime')
rides = pd.merge_asof(
    rides,
    weather,
    left_on='ride_start',
    right_on='datetime',
    direction='nearest'
)
print(rides.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4270 entries, 0 to 4269
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ride_type        4270 non-null   object        
 1   ride_category    4270 non-null   object        
 2   status           4270 non-null   object        
 3   tip              4270 non-null   float64       
 4   surge            4270 non-null   float64       
 5   duration         4270 non-null   int64         
 6   distance         4270 non-null   float64       
 7   pickup_address   4270 non-null   object        
 8   dropoff_address  4270 non-null   object        
 9   pickup_zip       4270 non-null   object        
 10  dropoff_zip      4269 non-null   object        
 11  earnings         4270 non-null   float64       
 12  base_pay         4270 non-null   float64       
 13  ride_start       4270 non-null   datetime64[ns]
 14  ride_end         4270 non-null   datetim

## Tip Distribution

Let's figure out the IQR and use that to determine outliers. 

In [3]:
Q1 = rides['tip'].quantile(0.25)
Q3 = rides['tip'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"Any tips outside the range ({lower_bound}, {upper_bound}) are outliers.")

Any tips outside the range (-4.5, 7.5) are outliers.


In [4]:
rides['tip_percent_base'] = 100 * rides['tip'] / (rides['base_pay'] + rides['surge'])

print(f"There are {len(rides)} total rides.")
for query in ('tip == 0', f"tip > 0 and tip <= {upper_bound}", f"tip > {upper_bound}"):
    subset = rides.query(query)
    subset_count = len(subset)
    tip_total = subset['tip'].sum()
    tip_avg = tip_total / subset_count
    distance_total = subset['distance'].sum()
    distance_avg = distance_total / subset_count
    earnings_total = subset['earnings'].sum()
    earnings_avg = earnings_total / subset_count
    base_total = subset['base_pay'].sum() + subset['surge'].sum()
    base_avg = base_total / subset_count
    tip_pct_base = 100 * subset['tip'].sum() / base_total
    print(f"There are {subset_count} rides ({100*subset_count/len(rides):.2f}%) with {query}.")
    print(f"\tI made ${tip_total:.2f} total tips and an average tip of ${tip_avg:.2f}/ride.")
    print(f"\tAverage trip base earnings+surge ${base_avg:.2f}/ride. On average, the tip was {tip_pct_base:.0f}% of this.")
    print(f"\tI drove a total of {distance_total:.2f} miles and an average of {distance_avg:.2f} miles/ride.")
    print(f"\tI made a total of ${earnings_total:.2f} on these rides, with an average of ${earnings_avg:.2f}/ride.")

rides['tip_given'] = (rides['tip'] > 0).astype(int)
rides['surge_active'] = (rides['surge'] > 0).astype(int)

There are 4270 total rides.
There are 2713 rides (63.54%) with tip == 0.
	I made $0.00 total tips and an average tip of $0.00/ride.
	Average trip base earnings+surge $9.82/ride. On average, the tip was 0% of this.
	I drove a total of 12286.10 miles and an average of 4.53 miles/ride.
	I made a total of $26632.19 on these rides, with an average of $9.82/ride.
There are 1437 rides (33.65%) with tip > 0 and tip <= 7.5.
	I made $5061.25 total tips and an average tip of $3.52/ride.
	Average trip base earnings+surge $9.76/ride. On average, the tip was 36% of this.
	I drove a total of 6186.20 miles and an average of 4.30 miles/ride.
	I made a total of $19092.70 on these rides, with an average of $13.29/ride.
There are 120 rides (2.81%) with tip > 7.5.
	I made $1321.43 total tips and an average tip of $11.01/ride.
	Average trip base earnings+surge $19.04/ride. On average, the tip was 58% of this.
	I drove a total of 1172.90 miles and an average of 9.77 miles/ride.
	I made a total of $3606.12 on

More than half of my rides got no tips, and a miniscule percentage of rides had tips over $7.50. I'm going to split my rides into three groups by tip amount:

1. `tip == 0`: Let's see what patterns there may be for the almost 2/3 of riders who don't tip.
2. `0 < tip < 7.5`: These are my stars. 
3. `tip >= 7.5`: There are only 120 high tippers out of 4270 rides. One contributing factor is that I didn't do a lot of very long trips, I mostly chose trips that kept me close to the busy areas. Something else weighting these is that I can remember a few riders who felt really touched by our conversation, and they tipped me highly for that more than the driving. 

## Zero Tipper Patterns

For rides where I got zero tip, is there a pattern? Considering that my average rating was between 4.98 and 5.00 at all times, I don't think it's the quality of my ride. Some plausible explanations I'll consider:

1. When surge pricing is high, riders may feel the fare is already inflated and choose not to add a tip.
2. Short durations or distances can lead to a perception that the service fee is sufficient, making a tip seem unnecessary.
3. Certain ride types (like UberX or UberX Share rides) might not encourage tipping as much as premium services (like Comfort or UberXL).
4. Adverse weather (reflected in fields like temp, feelslike, precip, or snow) might leave riders preoccupied or frustrated, reducing the likelihood of tipping.
5. If riders see that the base fare and overall earnings seem fair, they might not feel compelled to tip.
6. Socio-economic factors inferred from the pickup or dropoff addresses/zip codes might influence tipping behavior.
7. The timing of the ride (e.g., late at night or during off-peak hours) could correlate with lower tipping, possibly due to rider fatigue or different expectations.

Lastly, what I can't show with this data is that some riders simply have a no-tip habit regardless of the service provided.

In [5]:
# When surge pricing is high, riders may feel the fare is already inflated and choose not to add a tip.



In [11]:
# Short durations or distances can lead to a perception that the service fee is sufficient, making a tip seem unnecessary.
# Create dummy variables for ride_type (dropping one category to avoid multicollinearity)
ride_type_dummies = pd.get_dummies(rides['ride_type'], prefix='ride_type', drop_first=True)

# Combine predictors: ride duration, base_pay, feelslike, and the dummy variables for ride_type
X = pd.concat([rides[['duration', 'distance', 'base_pay', 'surge', 'feelslike']], ride_type_dummies], axis=1)
X = X.apply(pd.to_numeric)
# Add a constant term to include an intercept in the model
X = sm.add_constant(X)

# Outcome variable
y = rides['tip_given']

# Fit the logistic regression model
model = sm.Logit(y, X)
result = model.fit()

# Print the summary
print(result.summary())


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).